<a href="https://colab.research.google.com/github/ohki-yu0225/toukei_I/blob/main/261113_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### データ

本ノートブックで使用するデータは，Jリーグ公式サイトの「J.LEAGUE Data Site」の[試合記録・データ検索](https://data.j-league.or.jp/SFMS01/)から取得したJ1リーグの試合記録である。

- 取得元：https://data.j-league.or.jp/SFMS01/
- `2017-2025年J1試合記録_9チーム.csv` は，取得したデータから2017〜2025年（2020・2021年を除く）の試合のうち，この期間に常にJ1に所属していた9チームのホーム試合に絞り込んだものである。

In [ ]:
!pip install japanize_matplotlib

import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib

plt.rcParams["figure.figsize"] = (4, 3)
%config InlineBackend.figure_format = "retina"

file_id = "1MSihI6mLVVGaq-Xk8LwU-9Kkzyqf9Aoz"

gdown.download(
    id=file_id,
    output="2017-2025年J1試合記録_9チーム.csv",
    quiet=False
)

df = pd.read_csv("2017-2025年J1試合記録_9チーム.csv", encoding="utf-8-sig")

In [ ]:
# ホーム得点とアウェイ得点を比較して，ホームチームの勝ち・引き分け・負けを判定し，dfに追加
df["ホーム結果"] = np.where(df["ホーム得点"] > df["アウェイ得点"], "勝ち",
                  np.where(df["ホーム得点"] == df["アウェイ得点"], "引き分け", "負け"))

# ホーム得点を「0点・1点・2点・3点以上」にまとめた列を追加
df["ホーム得点区分"] = np.where(df["ホーム得点"] >= 3, "3点以上", df["ホーム得点"].astype(str) + "点")
df[["ホーム", "アウェイ", "ホーム得点", "アウェイ得点", "ホーム結果", "ホーム得点区分"]].head()

# 「土・祝」「月・休」など祝日・振替休日の試合は「祝日」にまとめる
df["曜日区分"] = np.where(df["曜日"].str.contains("祝|休"), "祝日", df["曜日"])
df

### 度数分布・ヒストグラム

In [ ]:
# 観客数の度数分布を作成する
class_width = 10000
bins = np.arange(0, df["入場者数"].max() + class_width, class_width)
freq = pd.cut(df["入場者数"], bins=bins, right=False).value_counts().sort_index()
freq_table = pd.DataFrame({
    "度数": freq,
    "相対度数": (freq / freq.sum()).round(3),
    "累積度数": freq.cumsum(),
})
freq_table

In [ ]:
# 観客数のヒストグラムを作成する
class_width = 10000
bins = np.arange(0, df["入場者数"].max() + class_width, class_width)
plt.hist(df["入場者数"], bins=bins, edgecolor="black")
plt.xlabel("入場者数（人）")
plt.ylabel("度数（試合数）")
plt.title("入場者数のヒストグラム")
plt.show()

In [ ]:
# 演習：階級の幅を変えて度数分布を作る
class_width = 10000
bins = np.arange(0, df["入場者数"].max() + class_width, class_width)
freq = pd.cut(df["入場者数"], bins=bins, right=False).value_counts().sort_index()
freq_table = pd.DataFrame({
    "度数": freq,
    "相対度数": (freq / freq.sum()).round(3),
    "累積度数": freq.cumsum(),
})
freq_table

In [ ]:
# 演習：階級の幅を変えてヒストグラムを作る
class_width = 10000
bins = np.arange(0, df["入場者数"].max() + class_width, class_width)
plt.hist(df["入場者数"], bins=bins, edgecolor="black")
plt.xlabel("入場者数（人）")
plt.ylabel("度数（試合数）")
plt.title("入場者数のヒストグラム")
plt.show()

### 代表値・広がりの尺度

In [ ]:
# 観客数の平均値・中央値を計算する
mean = df["入場者数"].mean()
median = df["入場者数"].median()
print(f"平均値: {mean:.1f}人")
print(f"中央値: {median:.1f}人")

In [ ]:
# 観客数の標準偏差・四分位範囲を計算する
std = df["入場者数"].std()
q1 = df["入場者数"].quantile(0.25)
q3 = df["入場者数"].quantile(0.75)
iqr = q3 - q1
print(f"標準偏差: {std:.1f}人")
print(f"第1四分位数: {q1:.1f}人, 第3四分位数: {q3:.1f}人")
print(f"四分位範囲: {iqr:.1f}人")

### 箱ひげ図

In [ ]:
# 観客数の箱ひげ図を作成する
sns.boxplot(y=df["入場者数"])
plt.ylabel("入場者数（人）")
plt.ylim(0, None)
plt.title("入場者数の箱ひげ図")
plt.show()

In [ ]:
# チームごとに観客数の箱ひげ図を作成する
sns.boxplot(data=df, x="ホーム", y="入場者数", order=None)
plt.xticks(rotation=90)
plt.xlabel("ホームチーム")
plt.ylabel("入場者数（人）")
plt.ylim(0, None)
plt.title("チームごとの入場者数（ホーム試合）")
plt.show()

In [ ]:
# 演習：曜日ごとに観客数の箱ひげ図を作成する
order = [d for d in ["月", "火", "水", "木", "金", "土", "日", "祝日"] if d in df["曜日区分"].unique()]


### クロス集計

In [ ]:
# チームごとにホーム勝ち数・引き分け数・負け数を集計するクロス集計
cross = pd.crosstab(df["ホーム"], df["ホーム結果"])
cross

In [ ]:
# 行和が100%になるように正規化（チームごとの勝率・引き分け率・負け率を計算）
cross_rate = pd.crosstab(df["ホーム"], df["ホーム結果"], normalize="index")
cross_rate

In [ ]:
# クロス集計表をヒートマップで可視化する
sns.heatmap(cross_rate, annot=True, fmt=".3f", cmap="Blues")
plt.xlabel("ホーム結果")
plt.ylabel("ホームチーム")
plt.title("チームごとのホームでの勝敗の割合")
plt.show()

In [ ]:
# 演習：チームごとにホーム得点を集計するクロス集計を作成する


In [ ]:
# 演習：行和が100%になるように正規化する


In [ ]:
# 演習：クロス集計表をヒートマップで可視化する


### 散布図・相関係数

In [ ]:
# チーム・年ごとにホーム勝率と観客数を計算する
team_stats = df.groupby(["ホーム", "シーズン"]).agg(
    試合数=("ホーム結果", "size"),
    ホーム勝率=("ホーム結果", lambda x: (x == "勝ち").mean()),
    平均入場者数=("入場者数", "mean"),
)

# チームごとの平均と，各年の値からチーム平均を引いた偏差を列に追加
team_stats["チーム平均ホーム勝率"] = team_stats.groupby(level="ホーム")["ホーム勝率"].transform("mean")
team_stats["チーム平均入場者数"] = team_stats.groupby(level="ホーム")["平均入場者数"].transform("mean")
team_stats["ホーム勝率の偏差"] = team_stats["ホーム勝率"] - team_stats["チーム平均ホーム勝率"]
team_stats["平均入場者数の偏差"] = team_stats["平均入場者数"] - team_stats["チーム平均入場者数"]
team_stats

In [ ]:
# チームごとにホーム勝率と観客数の平均値の関係を散布図で可視化する
plt.scatter(team_stats["ホーム勝率"], team_stats["平均入場者数"])
plt.xlabel("ホーム勝率")
plt.ylabel("平均入場者数（人）")
plt.title("ホーム勝率と平均入場者数")
plt.show()

In [ ]:
# チームごとにホーム勝率と観客数の平均値の相関関係を計算する
r = team_stats["ホーム勝率"].corr(team_stats["平均入場者数"])
print(f"相関係数: {r:.3f}")

In [ ]:
# 演習：チームごとの平均からの偏差を散布図で可視化する


In [ ]:
# 演習：チームごとの平均からの偏差の相関係数を計算する
